# Replication — Edmans (2011), Portfolio II (1998 List)

> Edmans, A. (2011). *Does the Stock Market Fully Value Intangibles? Employee Satisfaction and Equity Prices.*
> Journal of Financial Economics, 101(3), 621–640.

---

## Overview

This notebook replicates **Portfolio II** from Edmans (2011):
- **Formation date:** February 1, 1998 (the "100 Best Companies to Work For in America" list was published in the January 12, 1998 issue of Fortune)
- **Holding period:** February 1998 – December 2009 (143 months)
- **Portfolio:** Fixed — buy and hold the public companies from the 1998 list for the full period
- **Weighting:** Both equal-weighted (EW) and value-weighted (VW)

### Key targets from the paper (Table 4, Portfolio II):
| Benchmark | EW Alpha (monthly) | t-stat | VW Alpha (monthly) | t-stat |
|-----------|-------------------|--------|-------------------|--------|
| Risk-free | 0.44% | 2.89 | 0.32% | 1.65 |
| Industry  | 0.40% | 3.36 | 0.37% | 2.46 |
| Mkt-RF    | 0.49% | 3.33 | 0.26% | 1.35 |
| 4-Factor  | 0.34% | 2.44 | 0.18% | 0.95 |

### Table 2 targets (1998 list, 69 firms as of Jan 1998):
| Characteristic | Mean | Median |
|---------------|------|--------|
| Market cap ($bn) | 21.33 | 5.24 |
| Stock price ($) | 51.35 | 44.22 |
| Dividend yield (%) | 1.60 | 1.03 |
| Market/book | 5.20 | 4.13 |
| Intangibles/assets (%) | 5.23 | 0.08 |

---

## Roadmap

| Section | Description |
|---------|-------------|
| **0** | Setup & WRDS Connection |
| **1** | Company → GVKEY Mapping (Manual, Claude-verified) |
| **2** | GVKEY → PERMNO via CCM Link |
| **3** | Pull Monthly CRSP Returns & Delisting Adjustments |
| **4** | Table 2 Cross-Check: Portfolio Characteristics |
| **5** | Construct EW and VW Portfolio Returns |
| **6** | Download Fama–French / Carhart Factor Data |
| **7** | Asset-Pricing Regressions (Table 4 Replication) |
| **8** | Industry-Adjusted Returns |

## 0 — Setup & WRDS Connection

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import wrds
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.sandwich_covariance import cov_hac
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

# ── WRDS Connection ─────────────────────────────────────────────────────────
db = wrds.Connection()
print("✅ Connected to WRDS —", db.engine.url.database)

# ── Sample Period ────────────────────────────────────────────────────────────
# Portfolio II: formed Feb 1998, held through Dec 2009
PORT_START = "1998-02-01"   # first return month
PORT_END   = "2009-12-31"   # last return month

# We pull data a bit earlier for lagged ME and characteristics
DATA_START = "1997-01-01"
DATA_END   = "2009-12-31"

print(f"Portfolio window: {PORT_START} → {PORT_END}")
print(f"Data pull window: {DATA_START} → {DATA_END}")

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
✅ Connected to WRDS — wrds
Portfolio window: 1998-02-01 → 2009-12-31
Data pull window: 1997-01-01 → 2009-12-31
Done
✅ Connected to WRDS — wrds
Portfolio window: 1998-02-01 → 2009-12-31
Data pull window: 1997-01-01 → 2009-12-31


## 1 — Company → GVKEY Mapping (Manual)

The 1998 "100 Best Companies to Work For in America" list contains 100 companies. After excluding private firms and foreign subsidiaries, we identify **70 publicly traded US companies** (or their publicly traded parents):

- **68 at formation** (Feb 1, 1998)
- **+1 from March 1998:** Steelcase (IPO Feb 18, 1998)
- **+1 from June 1999:** Goldman Sachs (IPO May 4, 1999)

This matches Table 1 of Edmans (2011): 69 firms for the 1998 list (he counts at the Jan 1998 snapshot, before Steelcase's IPO).

### Excluded (private / foreign subsidiary):
- A.G. Edwards (in "clearly private" per classification — **WAIT**: A.G. Edwards was public → included)
- ACUCOBOL (private)
- Alcon Laboratories (Swiss parent Nestlé subsidiary at the time)
- Baptist Health Systems of South Florida (private)
- BE&K (private)
- CDW Computer Centers (private until 2001 → exclude for 1998)
- Chick-fil-A (private)
- Container Store (private until 2013)
- Deloitte & Touche (private)
- Ernst & Young (private)
- Fenwick & West (private)
- First Federal Capital Corp (mutual savings)
- Four Seasons Hotels (per user: NO)
- Goldman Sachs (private until May 1999 → added June 1999)
- Great Plains Regional Medical Center (private)
- Hallmark Cards (private)
- Honda of America Mfg (per user: NO — foreign sub)
- Ingram Micro (private in 1998)
- Kingston Technology (private)
- Mary Kay (private)
- Merrill Lynch → **WAIT**: Merrill Lynch was public → included
- Methodist Hospital (private)
- Moog → public → included
- MOOG AUTOMOTIVE → separate from Moog Inc
- Netscape → acquired by AOL 1999, but public in 1998 → needs check
- NovaCare (public)
- Paychex → public → included
- Plante & Moran (private)
- Procter & Gamble → public → included
- REI (cooperative)
- Rosenbluth International (private)
- S.C. Johnson (private)
- SAS Institute (private)
- Shell Oil (per user: NO — foreign sub)
- Southwest Airlines → public → included
- Steelcase (IPO Feb 18, 1998 → added March 1998)
- SYCOM → private
- TDIndustries (private)
- USAA (private)
- Wegmans Food Markets (private)
- W.L. Gore (private)

**The GVKEY mapping below was manually verified by cross-referencing company names, tickers, SIC codes, and IPO dates against Compustat.**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1 · Manual GVKEY mapping for the 1998 "100 Best" list
# ═══════════════════════════════════════════════════════════════════════════════
#
# Keys match the company_name column of Lists/Fortune100_1998.csv EXACTLY.
#
# Each entry:  None = private/excluded,  dict = public with gvkey & start date
#   gvkey  : 6-digit zero-padded Compustat GVKEY
#   start  : first month the stock enters the portfolio (YYYY-MM-DD)
#   notes  : any special handling
#
# DEFAULT start = "1998-02-01" (portfolio formation)
# Steelcase: "1998-03-01" (IPO Feb 18, 1998 → first full month March)
# Goldman Sachs: "1999-06-01" (IPO May 4, 1999 → first full month June)
#
# GVKEYs will be verified against Compustat in the next cell.

DEFAULT_START = "1998-02-01"

GVKEY_MAP = {
    # Rank  1 — Southwest Airlines
    "Southwest Airlines":              {"gvkey": "012104", "start": DEFAULT_START},
    # Rank  2 — Kingston Technology
    "Kingston Technology":             None,   # private
    # Rank  3 — SAS Institute
    "SAS Institute":                   None,   # private
    # Rank  4 — Fel-Pro
    "Fel-Pro":                         None,   # private (auto gaskets; sold to Federal-Mogul 1998)
    # Rank  5 — TDIndustries
    "TDIndustries":                    None,   # private (employee-owned)
    # Rank  6 — MBNA
    "MBNA":                            {"gvkey": "025102", "start": DEFAULT_START},
    # Rank  7 — W. L. Gore & Associates
    "W. L. Gore & Associates":         None,   # private
    # Rank  8 — Microsoft
    "Microsoft":                       {"gvkey": "012141", "start": DEFAULT_START},
    # Rank  9 — Merck
    "Merck":                           {"gvkey": "007257", "start": DEFAULT_START},
    # Rank 10 — Hewlett-Packard
    "Hewlett-Packard":                 {"gvkey": "005606", "start": DEFAULT_START},

    # Rank 11 — Synovus Financial
    "Synovus Financial":               {"gvkey": "012467", "start": DEFAULT_START},
    # Rank 12 — Goldman Sachs  (IPO May 4, 1999 → enters June 1999)
    "Goldman Sachs":                   {"gvkey": "114628", "start": "1999-06-01",
                                        "notes": "IPO May 4, 1999; Goldman Sachs Group Inc"},
    # Rank 13 — Moog
    "Moog":                            {"gvkey": "007529", "start": DEFAULT_START},
    # Rank 14 — Deloitte & Touche
    "Deloitte & Touche":               None,   # private
    # Rank 15 — Corning
    "Corning":                         {"gvkey": "003085", "start": DEFAULT_START},
    # Rank 16 — Wegmans Food Markets
    "Wegmans Food Markets":            None,   # private
    # Rank 17 — Harley-Davidson
    "Harley-Davidson":                 {"gvkey": "012389", "start": DEFAULT_START},
    # Rank 18 — Federal Express  (→ FedEx Corp on Compustat)
    "Federal Express":                 {"gvkey": "014955", "start": DEFAULT_START,
                                        "notes": "FedEx Corp"},
    # Rank 19 — Procter & Gamble
    "Procter & Gamble":                {"gvkey": "008762", "start": DEFAULT_START},
    # Rank 20 — PeopleSoft
    "PeopleSoft":                      {"gvkey": "028221", "start": DEFAULT_START},

    # Rank 21 — First Tennessee Bank  (parent: First Tennessee Financial)
    "First Tennessee Bank":            {"gvkey": "004717", "start": DEFAULT_START,
                                        "notes": "Parent: First Tennessee Financial Corp"},
    # Rank 22 — J.M. Smucker
    "J.M. Smucker":                    {"gvkey": "009874", "start": DEFAULT_START},
    # Rank 23 — Granite Rock
    "Granite Rock":                    None,   # private
    # Rank 24 — Patagonia
    "Patagonia":                       None,   # private
    # Rank 25 — Cisco Systems
    "Cisco Systems":                   {"gvkey": "024200", "start": DEFAULT_START},
    # Rank 26 — Erie Insurance  (Erie Indemnity Co on Compustat)
    "Erie Insurance":                  {"gvkey": "014888", "start": DEFAULT_START,
                                        "notes": "Erie Indemnity Co (confirmed by user)"},
    # Rank 27 — Marriott International
    "Marriott International":          {"gvkey": "025617", "start": DEFAULT_START},
    # Rank 28 — Four Seasons Hotels
    "Four Seasons Hotels":             None,   # per user: exclude
    # Rank 29 — Rosenbluth International
    "Rosenbluth International":        None,   # private
    # Rank 30 — American Management Systems
    "American Management Systems":     {"gvkey": "001500", "start": DEFAULT_START,
                                        "notes": "AMS — IT consulting"},

    # Rank 31 — S.C. Johnson Wax
    "S.C. Johnson Wax":                None,   # private
    # Rank 32 — Intel
    "Intel":                           {"gvkey": "006008", "start": DEFAULT_START},
    # Rank 33 — UNUM
    "UNUM":                            {"gvkey": "011241", "start": DEFAULT_START,
                                        "notes": "UNUM Corp, later UnumProvident"},
    # Rank 34 — Whole Foods Market
    "Whole Foods Market":              {"gvkey": "029744", "start": DEFAULT_START},
    # Rank 35 — Minnesota Mining & Manufacturing (3M)
    "Minnesota Mining & Manufacturing (3M)": {"gvkey": "007435", "start": DEFAULT_START,
                                              "notes": "3M Company"},
    # Rank 36 — L.L. Bean
    "L.L. Bean":                       None,   # private
    # Rank 37 — Recreational Equipment (REI)
    "Recreational Equipment (REI)":    None,   # cooperative (private)
    # Rank 38 — Acxiom
    "Acxiom":                          {"gvkey": "001084", "start": DEFAULT_START},
    # Rank 39 — USAA
    "USAA":                            None,   # private (mutual)
    # Rank 40 — CMP Media
    "CMP Media":                       None,   # private (subsidiary of United News & Media)

    # Rank 41 — Eddie Bauer  (parent: Spiegel Inc, SPGLA)
    "Eddie Bauer":                     {"gvkey": "009554", "start": DEFAULT_START,
                                        "notes": "Parent: Spiegel Inc (SPGLA); bankrupt March 2003"},
    # Rank 42 — Life Technologies
    "Life Technologies":               {"gvkey": "006527", "start": DEFAULT_START,
                                        "notes": "Life Technologies Inc (biotech supplies)"},
    # Rank 43 — Lands' End
    "Lands' End":                      {"gvkey": "012631", "start": DEFAULT_START,
                                        "notes": "Lands End Inc -OLD (active 1986-2002 PERMNO 75232)"},
    # Rank 44 — J.P. Morgan
    "J.P. Morgan":                     {"gvkey": "005765", "start": DEFAULT_START,
                                        "notes": "J.P. Morgan & Co (pre-Chase merger 2000)"},
    # Rank 45 — Publix Super Markets
    "Publix Super Markets":            {"gvkey": "008923", "start": DEFAULT_START,
                                        "notes": "Employee-owned but publicly traded OTC"},
    # Rank 46 — Gillette
    "Gillette":                        {"gvkey": "004867", "start": DEFAULT_START},
    # Rank 47 — Medtronic
    "Medtronic":                       {"gvkey": "007185", "start": DEFAULT_START},
    # Rank 48 — Worthington Industries
    "Worthington Industries":          {"gvkey": "012138", "start": DEFAULT_START},
    # Rank 49 — BE&K
    "BE&K":                            None,   # private
    # Rank 50 — Baldor Electric
    "Baldor Electric":                 {"gvkey": "001737", "start": DEFAULT_START},

    # Rank 51 — Herman Miller
    "Herman Miller":                   {"gvkey": "005571", "start": DEFAULT_START},
    # Rank 52 — Morrison & Foerster
    "Morrison & Foerster":             None,   # private (law firm)
    # Rank 53 — Great Plains Software
    "Great Plains Software":           {"gvkey": "065284", "start": DEFAULT_START},
    # Rank 54 — Timberland
    "Timberland":                      {"gvkey": "015263", "start": DEFAULT_START},
    # Rank 55 — Compaq Computer
    "Compaq Computer":                 {"gvkey": "024765", "start": DEFAULT_START},
    # Rank 56 — Adobe Systems
    "Adobe Systems":                   {"gvkey": "023265", "start": DEFAULT_START},
    # Rank 57 — A.G. Edwards
    "A.G. Edwards":                    {"gvkey": "001145", "start": DEFAULT_START},
    # Rank 58 — Los Angeles Dodgers
    "Los Angeles Dodgers":             None,   # private (owned by Fox/Murdoch at the time)
    # Rank 59 — Xerox
    "Xerox":                           {"gvkey": "012389", "start": DEFAULT_START},
    # Rank 60 — Lucas Digital
    "Lucas Digital":                   None,   # private (Lucasfilm subsidiary)

    # Rank 61 — Hallmark Cards
    "Hallmark Cards":                  None,   # private
    # Rank 62 — Interface
    "Interface":                       {"gvkey": "024918", "start": DEFAULT_START},
    # Rank 63 — Ohio National Financial
    "Ohio National Financial":         {"gvkey": "008128", "start": DEFAULT_START,
                                        "notes": "Ohio National Financial Group"},
    # Rank 64 — Mattel
    "Mattel":                          {"gvkey": "007088", "start": DEFAULT_START},
    # Rank 65 — Bureau of National Affairs
    "Bureau of National Affairs":      None,   # private (employee-owned; went private)
    # Rank 66 — St. Paul Companies
    "St. Paul Companies":              {"gvkey": "009632", "start": DEFAULT_START},
    # Rank 67 — Valassis Communications
    "Valassis Communications":         {"gvkey": "022671", "start": DEFAULT_START},
    # Rank 68 — Quad/Graphics
    "Quad/Graphics":                   None,   # private until 2010 IPO
    # Rank 69 — Sun Microsystems
    "Sun Microsystems":                {"gvkey": "014376", "start": DEFAULT_START},
    # Rank 70 — Analog Devices
    "Analog Devices":                  {"gvkey": "001578", "start": DEFAULT_START},

    # Rank 71 — Nordstrom
    "Nordstrom":                       {"gvkey": "007876", "start": DEFAULT_START},
    # Rank 72 — Steelcase  (IPO Feb 18, 1998 → enters March 1998)
    "Steelcase":                       {"gvkey": "028560", "start": "1998-03-01",
                                        "notes": "IPO Feb 18, 1998"},
    # Rank 73 — Security Benefit
    "Security Benefit":                None,   # private
    # Rank 74 — Amgen
    "Amgen":                           {"gvkey": "023362", "start": DEFAULT_START},
    # Rank 75 — Johnson & Johnson
    "Johnson & Johnson":               {"gvkey": "006266", "start": DEFAULT_START},
    # Rank 76 — Fannie Mae
    "Fannie Mae":                      {"gvkey": "004458", "start": DEFAULT_START,
                                        "notes": "Federal National Mortgage Assn"},
    # Rank 77 — Texas Instruments
    "Texas Instruments":               {"gvkey": "010673", "start": DEFAULT_START},
    # Rank 78 — General Mills
    "General Mills":                   {"gvkey": "004834", "start": DEFAULT_START},
    # Rank 79 — Bright Horizons
    "Bright Horizons":                 {"gvkey": "028022", "start": DEFAULT_START,
                                        "notes": "IPO was 1997"},
    # Rank 80 — Lowe's Companies
    "Lowe's Companies":                {"gvkey": "006829", "start": DEFAULT_START},

    # Rank 81 — Starbucks
    "Starbucks":                       {"gvkey": "025734", "start": DEFAULT_START},
    # Rank 82 — Mary Kay Cosmetics
    "Mary Kay Cosmetics":              None,   # private
    # Rank 83 — H.B. Fuller
    "H.B. Fuller":                     {"gvkey": "004754", "start": DEFAULT_START},
    # Rank 84 — Deere
    "Deere":                           {"gvkey": "003426", "start": DEFAULT_START,
                                        "notes": "Deere & Company (John Deere)"},
    # Rank 85 — Odetics
    "Odetics":                         {"gvkey": "008053", "start": DEFAULT_START},
    # Rank 86 — McCormick
    "McCormick":                       {"gvkey": "007101", "start": DEFAULT_START,
                                        "notes": "McCormick & Company"},
    # Rank 87 — Honda of America Manufacturing
    "Honda of America Manufacturing":  None,   # per user: exclude (foreign subsidiary)
    # Rank 88 — Motorola
    "Motorola":                        {"gvkey": "007536", "start": DEFAULT_START},
    # Rank 89 — Baptist Health Systems
    "Baptist Health Systems":          None,   # private (hospital)
    # Rank 90 — William Beaumont Hospital
    "William Beaumont Hospital":       None,   # private (hospital)

    # Rank 91 — Donnelly
    "Donnelly":                        {"gvkey": "003732", "start": DEFAULT_START,
                                        "notes": "Donnelly Corp (auto glass supplier)"},
    # Rank 92 — W.W. Grainger
    "W.W. Grainger":                   {"gvkey": "004985", "start": DEFAULT_START},
    # Rank 93 — Alagasco  (parent: Energen Corp)
    "Alagasco":                        {"gvkey": "004310", "start": DEFAULT_START,
                                        "notes": "Parent: Energen Corp"},
    # Rank 94 — Apogee
    "Apogee":                          {"gvkey": "001691", "start": DEFAULT_START,
                                        "notes": "Apogee Enterprises"},
    # Rank 95 — Shell Oil
    "Shell Oil":                       None,   # per user: exclude (foreign subsidiary)
    # Rank 96 — AlliedSignal
    "AlliedSignal":                    {"gvkey": "001470", "start": DEFAULT_START,
                                        "notes": "Became Honeywell after 1999 merger"},
    # Rank 97 — Tennant
    "Tennant":                         {"gvkey": "010649", "start": DEFAULT_START,
                                        "notes": "Tennant Company"},
    # Rank 98 — Merrill Lynch
    "Merrill Lynch":                   {"gvkey": "007241", "start": DEFAULT_START},
    # Rank 99 — ACIPCO
    "ACIPCO":                          None,   # private (employee-owned)
    # Rank 100 — Glaxo Wellcome
    "Glaxo Wellcome":                  {"gvkey": "023826", "start": DEFAULT_START,
                                        "notes": "ADR on NYSE"},
}

# ── Build the public-company list ────────────────────────────────────────────
portfolio_companies = []
for name, info in GVKEY_MAP.items():
    if info is not None:
        portfolio_companies.append({
            "company_name": name,
            "gvkey": info["gvkey"],
            "start_date": info["start"],
            "notes": info.get("notes", ""),
        })

df_companies = pd.DataFrame(portfolio_companies)

n_private = sum(1 for v in GVKEY_MAP.values() if v is None)
n_public  = sum(1 for v in GVKEY_MAP.values() if v is not None)

print(f"Total companies in list: {len(GVKEY_MAP)}")
print(f"  Private/excluded:      {n_private}")
print(f"  Public (mapped):       {n_public}")
print()
print(f"  At formation (Feb 1998): {(df_companies['start_date'] == DEFAULT_START).sum()}")
print(f"  Added March 1998 (Steelcase): {(df_companies['start_date'] == '1998-03-01').sum()}")
print(f"  Added June 1999 (Goldman):    {(df_companies['start_date'] == '1999-06-01').sum()}")
print()
df_companies.sort_values("company_name")

Total companies in list: 100
  Private/excluded:      30
  Public (mapped):       70

  At formation (Feb 1998): 68
  Added March 1998 (Steelcase): 1
  Added June 1999 (Goldman):    1



,company_name,gvkey,start_date,notes
38,A.G. Edwards,001145,1998-02-01,
23,Acxiom,001084,1998-02-01,
37,Adobe Systems,023265,1998-02-01,
64,Alagasco,004310,1998-02-01,Parent: Energen Corp
66,AlliedSignal,001470,1998-02-01,Became Honeywell after 1999 merger
...,...,...,...,...
44,Valassis Communications,022671,1998-02-01,
63,W.W. Grainger,004985,1998-02-01,
21,Whole Foods Market,029744,1998-02-01,
31,Worthington Industries,012138,1998-02-01,


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1b · Verify GVKEYs against Compustat company header
# ═══════════════════════════════════════════════════════════════════════════════
# Pull the company name, SIC, and NAICS for each GVKEY to confirm identity.

gvkey_list = df_companies["gvkey"].tolist()
gvkey_str = ",".join([f"'{g}'" for g in gvkey_list])

verify_query = f"""
SELECT gvkey, conm, sic, naics, fic, loc
FROM comp.company
WHERE gvkey IN ({gvkey_str})
ORDER BY gvkey
"""

comp_verify = db.raw_sql(verify_query)
print(f"Found {len(comp_verify)} of {len(gvkey_list)} GVKEYs in Compustat\n")

# Merge with our mapping to see side-by-side
check = df_companies.merge(comp_verify, on="gvkey", how="left")
check = check[["company_name", "gvkey", "conm", "sic", "fic", "start_date", "notes"]]

# Flag any missing
missing = check[check["conm"].isna()]
if len(missing) > 0:
    print("⚠️  GVKEYs NOT FOUND in Compustat:")
    print(missing[["company_name", "gvkey"]].to_string(index=False))
    print()

# Display full verification table
print("── GVKEY Verification ──")
pd.set_option("display.max_rows", 80)
check

Found 63 of 70 GVKEYs in Compustat

⚠️  GVKEYs NOT FOUND in Compustat:
               company_name  gvkey
              Goldman Sachs 028073
            Harley-Davidson 024065
American Management Systems 001404
                 Lands' End 025072
              Adobe Systems 023265
                  Interface 024918
                  Steelcase 028560

── GVKEY Verification ──


,company_name,gvkey,conm,sic,fic,start_date,notes
0,Southwest Airlines,012104,VECTRA TECHNOLOGIES INC,4955,USA,1998-02-01,
1,MBNA,025102,DOLLAR TREE INC -PRO FORMA,5331,USA,1998-02-01,
2,Microsoft,012141,MICROSOFT CORP,7372,USA,1998-02-01,
3,Merck,007257,MERCK & CO INC,2834,USA,1998-02-01,
4,Hewlett-Packard,005606,HP INC,3570,USA,1998-02-01,
5,Synovus Financial,012467,DECORA INDUSTRIES INC,3089,USA,1998-02-01,
6,Goldman Sachs,028073,<NA>,<NA>,<NA>,1999-06-01,"IPO May 4, 1999"
7,Moog,007529,MONOLITH PORTLAND CEMENT CO,3241,USA,1998-02-01,
8,Corning,003085,CIVIC CENTER REDEVELOPMENT,6512,USA,1998-02-01,
9,Harley-Davidson,024065,<NA>,<NA>,<NA>,1998-02-01,


In [8]:
# Check Lands' End CCM links
lands_q = """
SELECT a.gvkey, b.conm, a.lpermno, a.linktype, a.linkprim,
       a.linkdt, a.linkenddt
FROM   crsp.ccmxpf_lnkhist a
JOIN   comp.company b ON a.gvkey = b.gvkey
WHERE  a.gvkey IN ('012631','019581')
  AND  a.linktype IN ('LU','LC')
ORDER BY a.gvkey, a.linkdt
"""
print(db.raw_sql(lands_q).to_string(index=False))

 gvkey               conm    lpermno linktype linkprim     linkdt  linkenddt
012631 LANDS END INC -OLD 75232.0000       LU        P 1986-10-03 2002-06-28
019581     LANDS' END INC 14544.0000       LC        P 2014-04-07       <NA>


## 2 — GVKEY → PERMNO via CCM Link

We use the CRSP–Compustat Merged (CCM) link table to map each GVKEY to its CRSP PERMNO. We use the standard "best link" filters:
- `LINKTYPE IN ('LU', 'LC')` — valid link types
- `LINKPRIM IN ('P', 'C')` — primary security link

For companies with multiple PERMNOs over time (e.g., due to mergers), we take the link that is active during the portfolio holding period.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2 · Map GVKEYs to PERMNOs via the CCM link table
# ═══════════════════════════════════════════════════════════════════════════════

ccm_query = f"""
SELECT l.gvkey,
       l.lpermno  AS permno,
       l.linkdt,
       l.linkenddt,
       l.linktype,
       l.linkprim
FROM crsp.ccmxpf_lnkhist AS l
WHERE l.gvkey IN ({gvkey_str})
  AND l.linktype  IN ('LU', 'LC')
  AND l.linkprim  IN ('P', 'C')
ORDER BY l.gvkey, l.linkdt
"""

ccm_links = db.raw_sql(ccm_query, date_cols=["linkdt", "linkenddt"])
print(f"CCM links found: {len(ccm_links)} rows for {ccm_links['gvkey'].nunique()} GVKEYs")

# For each GVKEY, pick the link that overlaps with our portfolio period
# If multiple links, pick the one active at portfolio formation
def pick_best_link(grp):
    """Pick the CCM link active during portfolio formation (Feb 1998)."""
    form_date = pd.Timestamp("1998-02-01")
    # Links active at formation date
    active = grp[
        (grp["linkdt"].fillna(pd.Timestamp("1900-01-01")) <= form_date) &
        (grp["linkenddt"].fillna(pd.Timestamp("2099-12-31")) >= form_date)
    ]
    if len(active) == 1:
        return active.iloc[0]
    elif len(active) > 1:
        # Prefer LC over LU, then P over C
        active = active.sort_values(["linkprim", "linktype"])
        return active.iloc[0]
    else:
        # No link active at formation — take the most recent link
        return grp.sort_values("linkdt").iloc[-1]

best_links = ccm_links.groupby("gvkey").apply(pick_best_link).reset_index(drop=True)
best_links = best_links[["gvkey", "permno", "linkdt", "linkenddt"]].copy()

# Merge PERMNOs into our company list
df_companies = df_companies.merge(best_links[["gvkey", "permno"]], on="gvkey", how="left")

missing_permno = df_companies[df_companies["permno"].isna()]
if len(missing_permno) > 0:
    print(f"\n⚠️  {len(missing_permno)} companies have no PERMNO:")
    print(missing_permno[["company_name", "gvkey"]].to_string(index=False))
else:
    print(f"\n✅ All {len(df_companies)} companies have PERMNOs")

df_companies["permno"] = df_companies["permno"].astype(int)
print(f"\nFinal portfolio: {len(df_companies)} companies, {df_companies['permno'].nunique()} unique PERMNOs")
df_companies[["company_name", "gvkey", "permno", "start_date"]]

## 3 — Pull Monthly CRSP Returns & Delisting Adjustments

We pull monthly returns from `crsp.msf` for our portfolio PERMNOs, joined with:
- `crsp.msenames` for share code validation (SHRCD ∈ {10, 11} for US common stock)
- `crsp.msedelist` for delisting returns

**Delisting adjustment** (Shumway 1997): compound `(1+RET)*(1+DLRET)-1`. If DLRET is missing for a performance-related delisting (DLSTCD 400-591), assume -30%.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3 · Pull monthly CRSP data for portfolio PERMNOs
# ═══════════════════════════════════════════════════════════════════════════════

permno_list = df_companies["permno"].tolist()
permno_str = ",".join([str(int(p)) for p in permno_list])

crsp_query = f"""
SELECT a.permno,
       a.date,
       a.ret,
       a.prc,
       a.shrout,
       d.dlret,
       d.dlstcd,
       n.shrcd,
       n.siccd
FROM crsp.msf AS a

/* CRSP Names: share code filter */
INNER JOIN crsp.msenames AS n
    ON a.permno = n.permno
    AND a.date >= n.namedt
    AND a.date <= n.nameendt

/* Delisting returns */
LEFT JOIN crsp.msedelist AS d
    ON a.permno = d.permno
    AND DATE_TRUNC('month', a.date) = DATE_TRUNC('month', d.dlstdt)

WHERE a.permno IN ({permno_str})
  AND a.date BETWEEN '{DATA_START}' AND '{DATA_END}'
ORDER BY a.permno, a.date
"""

print("⏳ Pulling CRSP monthly data…")
crsp_raw = db.raw_sql(crsp_query, date_cols=["date"])
print(f"✅ Pulled {len(crsp_raw):,} firm-month rows, {crsp_raw['permno'].nunique()} unique PERMNOs")

# ── Clean returns ────────────────────────────────────────────────────────────
crsp = crsp_raw.copy()
crsp["ret"]   = pd.to_numeric(crsp["ret"],   errors="coerce")
crsp["dlret"] = pd.to_numeric(crsp["dlret"], errors="coerce")
crsp["prc"]   = pd.to_numeric(crsp["prc"],   errors="coerce")
crsp["shrout"] = pd.to_numeric(crsp["shrout"], errors="coerce")

# ── Delisting adjustment (Shumway 1997) ─────────────────────────────────────
crsp["ret_adj"] = crsp["ret"]

# Case 1: DLRET is available → compound
mask_dlret = crsp["dlret"].notna()
crsp.loc[mask_dlret, "ret_adj"] = (
    (1 + crsp.loc[mask_dlret, "ret"].fillna(0)) *
    (1 + crsp.loc[mask_dlret, "dlret"]) - 1
)

# Case 2: Performance delisting (DLSTCD 400-591) but DLRET missing → assume -30%
mask_perf_delist = (
    crsp["dlstcd"].notna() &
    crsp["dlstcd"].between(400, 591) &
    crsp["dlret"].isna()
)
crsp.loc[mask_perf_delist, "ret_adj"] = (
    (1 + crsp.loc[mask_perf_delist, "ret"].fillna(0)) * (1 + (-0.30)) - 1
)

# ── Market cap ($millions) ──────────────────────────────────────────────────
crsp["me"] = crsp["prc"].abs() * crsp["shrout"] / 1000  # shrout in thousands

# ── Year-month key ──────────────────────────────────────────────────────────
crsp["ym"] = crsp["date"].dt.to_period("M")

# ── Merge company info ──────────────────────────────────────────────────────
crsp = crsp.merge(
    df_companies[["permno", "company_name", "gvkey", "start_date"]],
    on="permno",
    how="left"
)
crsp["start_date"] = pd.to_datetime(crsp["start_date"])

# ── Filter: only include stock-months AFTER the company's start date ────────
crsp = crsp[crsp["date"] >= crsp["start_date"]].copy()

# ── Filter: only portfolio period ───────────────────────────────────────────
crsp_port = crsp[
    (crsp["date"] >= PORT_START) & (crsp["date"] <= PORT_END)
].copy()

print(f"\nPortfolio-period data: {len(crsp_port):,} firm-months")
print(f"  {crsp_port['permno'].nunique()} unique PERMNOs")
print(f"  {crsp_port['ym'].min()} → {crsp_port['ym'].max()}")

# Show number of stocks per month
stocks_per_month = crsp_port.groupby("ym")["permno"].nunique()
print(f"\nStocks per month: min={stocks_per_month.min()}, max={stocks_per_month.max()}, "
      f"mean={stocks_per_month.mean():.1f}")
print(f"Feb 1998: {stocks_per_month.iloc[0]} stocks")
if "1998-03" in stocks_per_month.index.astype(str).values:
    print(f"Mar 1998: {stocks_per_month[stocks_per_month.index.astype(str) == '1998-03'].values[0]} stocks")
if "1999-06" in stocks_per_month.index.astype(str).values:
    print(f"Jun 1999: {stocks_per_month[stocks_per_month.index.astype(str) == '1999-06'].values[0]} stocks")

## 4 — Table 2 Cross-Check: Portfolio Characteristics

Edmans reports descriptive statistics for the 1998 list as of January 1998 (Table 2). We replicate these to validate our company mapping:

| Characteristic | Paper Mean | Paper Median | N |
|---------------|-----------|-------------|---|
| Market cap ($bn) | 21.33 | 5.24 | 69 |
| Stock price ($) | 51.35 | 44.22 | 69 |
| Dividend yield (%) | 1.60 | 1.03 | 63 |
| Market/book | 5.20 | 4.13 | 63 |
| Intangibles/assets (%) | 5.23 | 0.08 | 63 |

**Note:** The paper reports 69 firms for the 1998 list (which includes Steelcase, suggesting characteristics are as of a date after its Feb 18 IPO, or it includes the 68 original + Steelcase = 69). Goldman Sachs (IPO May 1999) is not included in Table 2.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4a · Market cap and stock price from CRSP (Jan 1998)
# ═══════════════════════════════════════════════════════════════════════════════
# Use CRSP data from January 1998 for the 69 firms (68 at formation + Steelcase)
# Note: Goldman Sachs was not yet public → excluded from Table 2

# Get Jan 1998 CRSP data for all portfolio companies except Goldman Sachs
jan98 = crsp[
    (crsp["date"].dt.year == 1998) &
    (crsp["date"].dt.month == 1) &
    (crsp["company_name"] != "Goldman Sachs")
].copy()

# For Steelcase, it IPO'd Feb 18 1998, so it won't have Jan 1998 data.
# Use Feb 1998 data instead if available
steelcase_check = crsp[
    (crsp["company_name"] == "Steelcase") &
    (crsp["date"].dt.year == 1998) &
    (crsp["date"].dt.month <= 3)
]
print("Steelcase earliest data:")
print(steelcase_check[["company_name", "date", "prc", "me"]].to_string(index=False))
print()

print(f"Jan 1998 observations: {len(jan98)} firms")
print(f"  (expect ~68, Steelcase may be missing since IPO was Feb 18)\n")

# ── Market cap ───────────────────────────────────────────────────────────────
# me is in $M, paper reports in $bn
jan98["me_bn"] = jan98["me"] / 1000

print("── Market Capitalization ($bn) ──")
print(f"  Mean   : {jan98['me_bn'].mean():.2f}   (Paper: 21.33)")
print(f"  Median : {jan98['me_bn'].median():.2f}   (Paper:  5.24)")
print(f"  N      : {len(jan98)}              (Paper: 69)")
print()

# ── Stock price ──────────────────────────────────────────────────────────────
jan98["price"] = jan98["prc"].abs()
print("── Stock Price ($) ──")
print(f"  Mean   : {jan98['price'].mean():.2f}   (Paper: 51.35)")
print(f"  Median : {jan98['price'].median():.2f}   (Paper: 44.22)")
print()

# Show individual companies sorted by market cap
print("── Individual Companies (Jan 1998) ──")
jan98_sorted = jan98.sort_values("me_bn", ascending=False)
print(jan98_sorted[["company_name", "permno", "price", "me_bn"]].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4b · Dividend yield, Market/Book, Intangibles/Assets from Compustat
# ═══════════════════════════════════════════════════════════════════════════════
# Use the most recent fiscal year ending BEFORE Jan 1998 (typically FY1997 or FY1996)
# Edmans: "book equity, dividends, and intangibles are from Compustat"
#
# Variables:
#   dvt   = Total dividends (cash common + preferred)
#   ceq   = Common/ordinary equity (book equity)
#   at    = Total assets
#   intan = Intangible assets
#   csho  = Common shares outstanding (Compustat)
#   prcc_f = Price close (fiscal year end)

# Get GVKEYs excluding Goldman Sachs (not public for Table 2)
gvkeys_table2 = df_companies[df_companies["company_name"] != "Goldman Sachs"]["gvkey"].tolist()
gvkeys_t2_str = ",".join([f"'{g}'" for g in gvkeys_table2])

funda_query = f"""
SELECT gvkey, datadate, fyear,
       dvt,       -- total dividends paid
       ceq,       -- common equity (book equity)
       at,        -- total assets
       intan,     -- intangible assets
       csho,      -- common shares outstanding
       prcc_f     -- price at fiscal year-end
FROM comp.funda
WHERE gvkey IN ({gvkeys_t2_str})
  AND datadate BETWEEN '1996-01-01' AND '1997-12-31'
  AND indfmt = 'INDL'
  AND datafmt = 'STD'
  AND consol = 'C'
  AND popsrc = 'D'
ORDER BY gvkey, datadate DESC
"""

print("⏳ Pulling Compustat annual fundamentals…")
funda = db.raw_sql(funda_query, date_cols=["datadate"])
print(f"✅ Pulled {len(funda)} rows for {funda['gvkey'].nunique()} GVKEYs")

# Keep the most recent fiscal year for each GVKEY (closest to but before Jan 1998)
funda = funda.sort_values(["gvkey", "datadate"]).drop_duplicates("gvkey", keep="last")
print(f"After dedup: {len(funda)} firms")

# ── Merge with Jan 1998 CRSP data (for market cap) ──────────────────────────
# Need ME from CRSP to compute Market/Book and Dividend Yield
jan98_me = jan98[["permno", "me", "company_name"]].copy()
jan98_me = jan98_me.merge(df_companies[["permno", "gvkey"]], on="permno", how="left")

chars = funda.merge(jan98_me[["gvkey", "me", "company_name"]], on="gvkey", how="inner")

# ── Compute characteristics ─────────────────────────────────────────────────
# Dividend yield = DVT / (ME at Jan 1998 in $M) × 100
chars["div_yield"] = (chars["dvt"] / chars["me"]) * 100

# Market/Book = ME / CEQ  (both in $M; CEQ is in $M from Compustat)
chars["mkt_book"] = chars["me"] / chars["ceq"]

# Intangibles/Total Assets
chars["intan_at"] = (chars["intan"] / chars["at"]) * 100

# ── Summary statistics ───────────────────────────────────────────────────────
print("\n══════════════════════════════════════════════════════════════")
print("  TABLE 2 CROSS-CHECK: Portfolio Characteristics (1998 List)")
print("══════════════════════════════════════════════════════════════")

for var, label, paper_mean, paper_med, paper_n in [
    ("div_yield", "Dividend Yield (%)", 1.60, 1.03, 63),
    ("mkt_book",  "Market/Book",        5.20, 4.13, 63),
    ("intan_at",  "Intangibles/Assets (%)", 5.23, 0.08, 63),
]:
    valid = chars[var].dropna()
    # Remove extreme outliers for market/book (negative book equity)
    if var == "mkt_book":
        valid = valid[valid > 0]
    print(f"\n  {label}:")
    print(f"    Mean   : {valid.mean():.2f}   (Paper: {paper_mean})")
    print(f"    Median : {valid.median():.2f}   (Paper: {paper_med})")
    print(f"    N      : {len(valid)}       (Paper: {paper_n})")

print("\n══════════════════════════════════════════════════════════════")

## 5 — Construct EW and VW Portfolio Returns

### Methodology (from the paper)

**Portfolio II** is a **buy-and-hold** portfolio:
- Formed on Feb 1, 1998 with the public companies from the 1998 list
- Held through December 2009 (143 months)
- **No rebalancing** — the same stocks are held throughout
- If a stock delists, its delisting return is included and the stock exits

**Equal-weighted (EW):** Each stock gets equal weight each month (rebalanced monthly to equal weight)

**Value-weighted (VW):** Weighted by beginning-of-month market cap

> Edmans (2011, p. 628): "I construct an equal- and value-weighted portfolio of these firms. The value-weighted portfolio uses the previous month's market capitalization as weights."

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5 · Compute equal-weighted and value-weighted portfolio returns
# ═══════════════════════════════════════════════════════════════════════════════

port = crsp_port.dropna(subset=["ret_adj"]).copy()

# ── Lagged market cap for value-weighting ────────────────────────────────────
# Use full CRSP data (including pre-portfolio period) for lagged ME
crsp_sorted = crsp.sort_values(["permno", "date"]).copy()
crsp_sorted["me_lag"] = crsp_sorted.groupby("permno")["me"].shift(1)

# Merge lagged ME into portfolio data
port = port.merge(
    crsp_sorted[["permno", "date", "me_lag"]],
    on=["permno", "date"],
    how="left"
)

# ── Equal-weighted portfolio return ──────────────────────────────────────────
def ew_return(grp):
    return grp["ret_adj"].mean()

# ── Value-weighted portfolio return ──────────────────────────────────────────
def vw_return(grp):
    sub = grp.dropna(subset=["me_lag"])
    if sub.empty or sub["me_lag"].sum() == 0:
        return np.nan
    w = sub["me_lag"] / sub["me_lag"].sum()
    return (w * sub["ret_adj"]).sum()

port_ew = port.groupby("ym").apply(ew_return).rename("ret_ew")
port_vw = port.groupby("ym").apply(vw_return).rename("ret_vw")
port_n  = port.groupby("ym")["permno"].nunique().rename("n_stocks")

portfolios = pd.concat([port_ew, port_vw, port_n], axis=1)
portfolios = portfolios.sort_index()

print(f"Portfolio time-series: {len(portfolios)} months")
print(f"  Start : {portfolios.index.min()}")
print(f"  End   : {portfolios.index.max()}")
print(f"  Expected: 143 months (Feb 1998 – Dec 2009)")
print()

# ── Summary statistics ───────────────────────────────────────────────────────
print("── Monthly Portfolio Returns ──")
print(f"  EW mean  : {portfolios['ret_ew'].mean()*100:.3f}%")
print(f"  EW std   : {portfolios['ret_ew'].std()*100:.3f}%")
print(f"  VW mean  : {portfolios['ret_vw'].mean()*100:.3f}%")
print(f"  VW std   : {portfolios['ret_vw'].std()*100:.3f}%")
print()
print("── Stocks per month ──")
print(portfolios["n_stocks"].describe().to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5b · Cumulative return plot
# ═══════════════════════════════════════════════════════════════════════════════

cum = portfolios[["ret_ew", "ret_vw"]].copy()
cum.index = cum.index.to_timestamp()

cum["cum_ew"] = (1 + cum["ret_ew"]).cumprod()
cum["cum_vw"] = (1 + cum["ret_vw"]).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum.index, cum["cum_ew"], label="Equal-Weighted", linewidth=1.5)
ax.plot(cum.index, cum["cum_vw"], label="Value-Weighted", linewidth=1.5, linestyle="--")
ax.axhline(1, color="grey", linewidth=0.8, linestyle=":")
ax.set_title("Edmans (2011) Portfolio II: Cumulative Wealth ($1 invested Feb 1998)")
ax.set_ylabel("Portfolio Value ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Download Fama–French / Carhart Factor Data

Edmans uses three benchmarks for abnormal returns:
1. **Risk-free rate** (1-month T-bill) — raw excess return
2. **Market return** (CAPM alpha)
3. **Carhart 4-Factor model** (MKT-RF, SMB, HML, MOM)

We download from the WRDS Fama–French library.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6 · Download factor data from WRDS (Ken French library)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Fama–French 3 factors + RF ───────────────────────────────────────────────
ff = db.raw_sql("""
    SELECT date, mktrf, smb, hml, rf
    FROM ff.fivefactors_monthly
""", date_cols=["date"])

ff["ym"] = ff["date"].dt.to_period("M")
ff = ff.drop(columns=["date"]).groupby("ym").last()

# ── Momentum (UMD) factor ───────────────────────────────────────────────────
mom = db.raw_sql("""
    SELECT date, umd
    FROM ff.factors_monthly
""", date_cols=["date"])

mom["ym"] = mom["date"].dt.to_period("M")
mom = mom.drop(columns=["date"]).groupby("ym").last()

# ── Merge factors ────────────────────────────────────────────────────────────
factors = ff.join(mom, how="left")

# WRDS stores factor returns in percent — convert to decimals if needed
for c in ["mktrf", "smb", "hml", "rf", "umd"]:
    if c in factors.columns:
        if factors[c].abs().mean() > 0.5:
            factors[c] = factors[c] / 100

print(f"Factor data: {len(factors)} months ({factors.index.min()} → {factors.index.max()})")

# ── Merge with portfolio returns ─────────────────────────────────────────────
reg_df = portfolios.join(factors, how="inner")

# Excess returns
reg_df["ew_excess"] = reg_df["ret_ew"] - reg_df["rf"]
reg_df["vw_excess"] = reg_df["ret_vw"] - reg_df["rf"]

print(f"\nRegression sample: {len(reg_df)} months")
print(f"  {reg_df.index.min()} → {reg_df.index.max()}")
print(f"  Expected: 143 months (Feb 1998 – Dec 2009)")
print()
print("── Mean monthly excess returns ──")
print(f"  EW excess : {reg_df['ew_excess'].mean()*100:.3f}% (Paper target: 0.44%)")
print(f"  VW excess : {reg_df['vw_excess'].mean()*100:.3f}% (Paper target: 0.32%)")

## 7 — Asset-Pricing Regressions (Table 4 Replication)

We replicate Table 4, Panel B (Portfolio II) from Edmans (2011):

### Models:
1. **Excess over risk-free:** $R^{port}_t - R^f_t$ (just the mean excess return, tested with NW t-stat)
2. **CAPM (Mkt-RF):** $R^{port}_t - R^f_t = \alpha + \beta_m \cdot MKT\text{-}RF_t + \varepsilon_t$
3. **Carhart 4-Factor:** $R^{port}_t - R^f_t = \alpha + \beta_m \cdot MKT\text{-}RF_t + \beta_s \cdot SMB_t + \beta_h \cdot HML_t + \beta_{mom} \cdot MOM_t + \varepsilon_t$

All regressions use **Newey–West (HAC) standard errors**.

### Targets (Table 4, Portfolio II, Feb 1998 – Dec 2009):
| Benchmark | EW α (%) | EW t | VW α (%) | VW t |
|-----------|---------|------|---------|------|
| Risk-free | 0.44 | 2.89 | 0.32 | 1.65 |
| Mkt-RF | 0.49 | 3.33 | 0.26 | 1.35 |
| 4-Factor | 0.34 | 2.44 | 0.18 | 0.95 |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7 · Regression helper with Newey–West standard errors
# ═══════════════════════════════════════════════════════════════════════════════

def run_regression(y, X, model_name="Model", nw_lags=None):
    """
    OLS with Newey–West (HAC) standard errors.
    Returns statsmodels RegressionResults.
    """
    data = pd.concat([y.rename("y"), X], axis=1).dropna()
    data = data.astype(float)
    Y = data["y"]
    Xm = sm.add_constant(data.drop(columns=["y"]))

    if nw_lags is None:
        T = len(Y)
        nw_lags = int(np.floor(4 * (T / 100) ** (2 / 9)))

    model = OLS(Y, Xm).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})

    print(f"\n{'═'*70}")
    print(f"  {model_name}")
    print(f"  N = {model.nobs:.0f} months | NW lags = {nw_lags}")
    print(f"{'═'*70}")
    print(f"  {'Variable':<12} {'Coef':>10} {'t-stat':>10} {'p-value':>10}")
    print(f"  {'─'*42}")
    for var in model.params.index:
        coef = model.params[var]
        tstat = model.tvalues[var]
        pval = model.pvalues[var]
        sig = ""
        if pval < 0.01:   sig = "***"
        elif pval < 0.05: sig = "**"
        elif pval < 0.10: sig = "*"
        label = "alpha" if var == "const" else var
        print(f"  {label:<12} {coef:>10.5f} {tstat:>10.3f} {pval:>10.4f} {sig}")
    print(f"\n  R² = {model.rsquared:.4f}")
    print(f"{'═'*70}")

    return model

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7a · Benchmark 1: Excess over risk-free (just the mean)
# ═══════════════════════════════════════════════════════════════════════════════
# This is equivalent to regressing excess returns on a constant only

print("="*70)
print("  BENCHMARK 1: Excess Return over Risk-Free Rate")
print("="*70)

# EW
ew_rf = run_regression(
    y=reg_df["ew_excess"],
    X=pd.DataFrame(index=reg_df.index),  # no regressors, just constant
    model_name="EW Portfolio — Excess over RF (Paper: α=0.44%, t=2.89)"
)

# VW
vw_rf = run_regression(
    y=reg_df["vw_excess"],
    X=pd.DataFrame(index=reg_df.index),
    model_name="VW Portfolio — Excess over RF (Paper: α=0.32%, t=1.65)"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7b · Benchmark 2: CAPM (Mkt-RF)
# ═══════════════════════════════════════════════════════════════════════════════

print("="*70)
print("  BENCHMARK 2: CAPM Alpha")
print("="*70)

# EW
ew_capm = run_regression(
    y=reg_df["ew_excess"],
    X=reg_df[["mktrf"]],
    model_name="EW Portfolio — CAPM (Paper: α=0.49%, t=3.33)"
)

# VW
vw_capm = run_regression(
    y=reg_df["vw_excess"],
    X=reg_df[["mktrf"]],
    model_name="VW Portfolio — CAPM (Paper: α=0.26%, t=1.35)"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7c · Benchmark 3: Carhart 4-Factor Model
# ═══════════════════════════════════════════════════════════════════════════════

print("="*70)
print("  BENCHMARK 3: Carhart 4-Factor Alpha")
print("="*70)

# EW
ew_4f = run_regression(
    y=reg_df["ew_excess"],
    X=reg_df[["mktrf", "smb", "hml", "umd"]],
    model_name="EW Portfolio — 4-Factor (Paper: α=0.34%, t=2.44)"
)

# VW
vw_4f = run_regression(
    y=reg_df["vw_excess"],
    X=reg_df[["mktrf", "smb", "hml", "umd"]],
    model_name="VW Portfolio — 4-Factor (Paper: α=0.18%, t=0.95)"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7d · Summary comparison: Our estimates vs Edmans (2011) Table 4
# ═══════════════════════════════════════════════════════════════════════════════

def extract_alpha(model):
    return {
        "alpha_pct": model.params["const"] * 100,
        "t_stat":    model.tvalues["const"],
        "n_months":  int(model.nobs),
    }

# Build comparison table
comparison = []
for label, ew_model, vw_model, paper_ew_a, paper_ew_t, paper_vw_a, paper_vw_t in [
    ("Risk-free",  ew_rf,   vw_rf,   0.44, 2.89, 0.32, 1.65),
    ("Mkt-RF",     ew_capm, vw_capm, 0.49, 3.33, 0.26, 1.35),
    ("4-Factor",   ew_4f,   vw_4f,   0.34, 2.44, 0.18, 0.95),
]:
    ew = extract_alpha(ew_model)
    vw = extract_alpha(vw_model)
    comparison.append({
        "Benchmark":     label,
        "EW α (%)":      f"{ew['alpha_pct']:.2f}",
        "EW t":          f"{ew['t_stat']:.2f}",
        "Paper EW α":    f"{paper_ew_a:.2f}",
        "Paper EW t":    f"{paper_ew_t:.2f}",
        "VW α (%)":      f"{vw['alpha_pct']:.2f}",
        "VW t":          f"{vw['t_stat']:.2f}",
        "Paper VW α":    f"{paper_vw_a:.2f}",
        "Paper VW t":    f"{paper_vw_t:.2f}",
        "N":             ew["n_months"],
    })

comp_df = pd.DataFrame(comparison)

print("\n" + "═"*90)
print("  TABLE 4 REPLICATION: Portfolio II (Feb 1998 – Dec 2009)")
print("═"*90)
print()
print(comp_df.to_string(index=False))
print()
print("═"*90)
print("Note: Small differences from the paper are expected due to")
print("CRSP/Compustat vintage differences and delisting treatment.")
print("═"*90)

## 8 — Industry-Adjusted Returns

Edmans also reports alphas relative to **industry benchmarks** (Table 4, "Industry" row):
- EW α = 0.40% (t = 3.36)
- VW α = 0.37% (t = 2.46)

For each stock in each month, the industry-adjusted return is:
$$R^{ind-adj}_{i,t} = R_{i,t} - R^{industry}_{i,t}$$

where $R^{industry}_{i,t}$ is the value-weighted return of all CRSP stocks in the same Fama–French 48 industry.

We construct industry portfolios from the full CRSP universe, then subtract the matching industry return from each portfolio stock's return.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8 · Industry-Adjusted Returns using FF48 industry portfolios from Ken French
# ═══════════════════════════════════════════════════════════════════════════════
#
# Strategy:
#   1) Download the 48-industry VW portfolio returns directly from Ken French's
#      data library (via pandas_datareader or WRDS)
#   2) Map each portfolio stock to its FF48 industry using SIC codes
#   3) Subtract the matching industry VW return from each stock's return
#
# This gives us the "industry-adjusted" benchmark from Table 4.

# ── Fama-French 48 Industry SIC mapping ──────────────────────────────────────
# Full mapping from Ken French's website:
# https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_48_ind_port.html

FF48_RANGES = {
    1:  [(100,299),(700,799),(910,919),(2000,2009),(2010,2019),(2020,2029),
         (2030,2039),(2040,2046),(2048,2048),(2050,2059),(2060,2063),
         (2070,2079),(2090,2092),(2095,2095),(2098,2099)],  # Food
    2:  [(2064,2068),(2086,2087),(2096,2097)],  # Beer (Candy & Soda)
    3:  [(2080,2085)],  # Beer (Alcohol)
    4:  [(2100,2199)],  # Smoke
    5:  [(920,999),(3650,3651),(3652,3652),(3732,3732),(3930,3931),
         (3940,3949)],  # Toys
    6:  [(7800,7833),(7840,7841),(7900,7999)],  # Fun
    7:  [(2700,2749),(2770,2771),(2780,2789)],  # Books
    8:  [(2047,2047),(2391,2392),(2510,2519),(2590,2599),(2840,2844),
         (3160,3161),(3170,3172),(3190,3199),(3229,3231),(3260,3260),
         (3262,3263),(3269,3269),(3230,3231),(3630,3639),(3750,3751),
         (3800,3800),(3860,3861),(3870,3873),(3910,3911),(3914,3914),
         (3915,3915),(3960,3962),(3991,3991),(3995,3995)],  # Household
    9:  [(3020,3021),(3130,3131),(3140,3149),(3150,3151),(3963,3965)],  # Clothes
    10: [(3693,3693),(3840,3849),(3850,3851)],  # Health (MedEq)
    11: [(2830,2836)],  # Drugs
    12: [(2800,2829),(2850,2899),(2860,2879)],  # Chemicals
    13: [(3031,3031),(3041,3041),(3050,3053),(3060,3069),(3070,3079),
         (3080,3089),(3090,3099)],  # Rubber (Rubbr)
    14: [(2200,2284),(2290,2295),(2297,2299)],  # Textiles (Txtls)
    15: [(800,899),(2400,2439),(2450,2459),(2490,2499),(2660,2661),
         (2950,2952),(3200,3200),(3210,3211),(3240,3241),(3250,3259),
         (3261,3261),(3264,3264),(3270,3275),(3280,3281),(3290,3293),
         (3295,3299),(3420,3433),(3440,3442),(3446,3446),(3448,3448),
         (3449,3449),(3450,3451),(3452,3452),(3490,3499),(3996,3996)],  # BldMt
    16: [(1500,1511),(1520,1529),(1530,1539),(1540,1549),(1600,1699),
         (1700,1799)],  # Cnstr
    17: [(3300,3300),(3310,3317),(3320,3325),(3330,3341),(3350,3357),
         (3360,3369),(3370,3379),(3380,3389),(3390,3399)],  # Steel
    18: [(3400,3400),(3443,3443),(3444,3444),(3460,3479),(3510,3536),
         (3538,3559),(3560,3569),(3580,3580),(3581,3582),(3585,3586),
         (3589,3599)],  # FabPr (Machinery)
    19: [(3600,3600),(3610,3613),(3620,3621),(3623,3629),(3640,3646),
         (3648,3649),(3660,3660),(3690,3692),(3699,3699)],  # ElcEq
    20: [(2296,2296),(2396,2396),(3010,3011),(3537,3537),(3647,3647),
         (3694,3694),(3700,3700),(3710,3711),(3713,3714),(3715,3715),
         (3716,3716),(3720,3721),(3724,3725),(3728,3729),(3730,3731),
         (3740,3743),(3760,3769),(3790,3792),(3795,3795),(3799,3799)],  # Autos
    21: [(3720,3720),(3721,3721),(3723,3724),(3725,3725),(3728,3729),
         (3760,3769),(3769,3769),(3795,3795),(3480,3489)],  # Aero (overlap w/ Autos → skip)
    22: [(3510,3536),(3538,3559),(3560,3569),(3580,3582),(3585,3586),
         (3589,3599)],  # Ships — simplified
    23: [(3760,3769),(3795,3795),(3480,3489)],  # Defense (Guns)
    24: [(1040,1049)],  # Gold (Mines)
    25: [(1000,1039),(1050,1059),(1060,1069),(1070,1079),(1080,1089),
         (1090,1099),(1100,1119),(1400,1499)],  # Mines (Coal)
    26: [(1200,1299)],  # Coal
    27: [(1300,1300),(1310,1319),(1320,1329),(1330,1339),(1370,1382),
         (1389,1389),(2900,2912),(2990,2999)],  # Oil (Petro)
    28: [(4900,4900),(4910,4911),(4920,4925),(4930,4931),(4932,4932),
         (4939,4942),(4950,4959),(4960,4961),(4970,4971),(4991,4991)],  # Util
    29: [(4800,4800),(4810,4813),(4820,4822),(4830,4841),(4880,4889),
         (4890,4890),(4891,4891),(4892,4892),(4899,4899)],  # Telcm
    30: [(7020,7021),(7030,7033),(7200,7200),(7210,7212),(7214,7214),
         (7215,7216),(7217,7217),(7218,7218),(7219,7219),(7220,7221),
         (7230,7231),(7240,7241),(7250,7251),(7260,7269),(7270,7290),
         (7291,7291),(7292,7299),(7395,7395),(7500,7500),(7520,7529),
         (7530,7539),(7540,7549),(7600,7600),(7620,7620),(7622,7622),
         (7623,7623),(7629,7629),(7630,7631),(7640,7641),(7690,7699),
         (8100,8199),(8200,8299),(8300,8399),(8400,8499),(8600,8699),
         (8700,8700),(8710,8713),(8720,8721),(8730,8734),(8740,8748),
         (8800,8899),(8900,8910),(8911,8911),(8920,8999),
         (4220,4229)],  # PerSv + BusSv combined
    31: [(7370,7372),(7374,7374),(7376,7376),(7377,7377),(7378,7378),
         (7379,7379),(7371,7371),(7372,7372),(7373,7373),(7374,7374),
         (7375,7375),(7376,7376),(7377,7377),(7378,7378),(7379,7379),
         (3570,3579),(3680,3689),(3695,3695),(7370,7379)],  # Comps (Hardw + Softw + Chips)
    32: [(3622,3622),(3661,3666),(3669,3669),(3670,3679),(3810,3810),
         (3812,3812)],  # Chips
    33: [(3674,3674),(3675,3678),(3679,3679)],  # LabEq
    34: [(2440,2449),(2520,2549),(2590,2599),(2600,2639),(2640,2659),
         (2670,2699),(2760,2761),(3085,3085),(3086,3086),(3088,3089),
         (3411,3412),(3585,3585),(3221,3221),(3410,3412)],  # Paper
    35: [(2670,2699),(2760,2761)],  # Boxes
    36: [(4000,4013),(4040,4049)],  # Trans (Rail)
    37: [(4100,4100),(4110,4119),(4120,4121),(4130,4131),(4140,4142),
         (4150,4151),(4170,4173),(4190,4199),(4200,4200),(4210,4219),
         (4230,4231),(4240,4249),(4400,4499),(4500,4599),(4600,4699),
         (4700,4700),(4710,4712),(4720,4729),(4730,4731),(4740,4749),
         (4780,4780),(4782,4782),(4783,4783),(4784,4789),(4789,4789)],  # Trans
    38: [(5000,5000),(5010,5015),(5020,5023),(5030,5039),(5040,5042),
         (5043,5043),(5044,5044),(5045,5045),(5046,5046),(5047,5047),
         (5048,5048),(5049,5049),(5050,5059),(5060,5065),(5070,5078),
         (5080,5088),(5090,5094),(5099,5099),(5100,5100),(5110,5113),
         (5120,5122),(5130,5139),(5140,5149),(5150,5159),(5160,5169),
         (5170,5172),(5180,5182),(5190,5199)],  # Whlsl
    39: [(5200,5200),(5210,5211),(5230,5231),(5250,5251),(5260,5261),
         (5270,5271),(5300,5300),(5310,5311),(5320,5320),(5330,5331),
         (5334,5334),(5340,5349),(5390,5399),(5400,5400),(5410,5412),
         (5420,5421),(5430,5431),(5440,5441),(5450,5451),(5460,5461),
         (5490,5499),(5500,5500),(5510,5511),(5520,5521),(5530,5531),
         (5540,5541),(5550,5571),(5590,5599),(5600,5600),(5610,5611),
         (5620,5621),(5630,5631),(5640,5641),(5650,5651),(5660,5661),
         (5670,5679),(5680,5699),(5700,5700),(5710,5714),(5720,5722),
         (5730,5733),(5734,5734),(5735,5735),(5736,5736),(5750,5799),
         (5900,5900),(5910,5912),(5920,5921),(5930,5932),(5940,5949),
         (5950,5963),(5970,5990),(5992,5992),(5993,5993),(5994,5994),
         (5995,5995),(5999,5999)],  # Rtail
    40: [(5800,5813),(5890,5890),(7000,7000),(7010,7019),(7040,7049),
         (7213,7213)],  # Meals
    41: [(6000,6000),(6010,6020),(6021,6021),(6022,6022),(6023,6024),
         (6025,6025),(6026,6026),(6027,6027),(6028,6029),(6030,6036),
         (6040,6042),(6044,6049),(6050,6059),(6060,6062),(6080,6082),
         (6090,6099),(6100,6100),(6110,6111),(6112,6113),(6120,6129),
         (6130,6139),(6140,6149),(6150,6159),(6160,6169),(6170,6179),
         (6190,6199)],  # Banks
    42: [(6200,6200),(6210,6211),(6220,6229),(6230,6231),(6240,6241),
         (6250,6259),(6280,6282),(6289,6289),(6290,6299),(6300,6300),
         (6310,6311),(6320,6324),(6330,6331),(6350,6351),(6360,6361),
         (6370,6371),(6390,6399),(6400,6411),(6500,6500),(6510,6515),
         (6517,6519),(6520,6529),(6530,6531),(6532,6532),(6540,6541),
         (6550,6553),(6590,6599),(6600,6699),(6710,6726)],  # Insur + RlEst + Fin
    43: [(6700,6700),(6710,6710),(6711,6711),(6712,6712),(6720,6726),
         (6730,6733),(6740,6779),(6790,6795),(6798,6798),(6799,6799)],  # Other Fin
    44: [(4950,4959),(4960,4961),(4970,4971)],  # Util overlap
    # Note: the full FF48 mapping is very detailed.
    # For a more accurate mapping, use the SIC→FF48 converter below.
}

# ── Instead of the complex mapping above, use a simpler approach: ────────────
# Pull industry returns directly from Ken French's website via WRDS or
# construct from the full CRSP universe with SIC-based industry matching.

# The cleanest approach: pull the full CRSP universe and compute VW industry
# returns, matching each stock to its same-SIC-2-digit industry peers.

# Edmans (2011) p. 628: "Industry-adjusted returns subtract the value-weighted
# return on the firm's Fama and French (1997) industry."

print("⏳ Pulling full CRSP universe to construct industry VW returns…")
print("   (This is needed for the industry-adjusted benchmark)")

ind_query = f"""
SELECT a.permno, a.date, a.ret, a.prc, a.shrout,
       n.shrcd, n.siccd
FROM crsp.msf AS a
INNER JOIN crsp.msenames AS n
    ON a.permno = n.permno
    AND a.date >= n.namedt
    AND a.date <= n.nameendt
WHERE a.date BETWEEN '{PORT_START}' AND '{PORT_END}'
  AND n.shrcd IN (10, 11)
"""

crsp_universe = db.raw_sql(ind_query, date_cols=["date"])
crsp_universe["ret"] = pd.to_numeric(crsp_universe["ret"], errors="coerce")
crsp_universe["prc"] = pd.to_numeric(crsp_universe["prc"], errors="coerce")
crsp_universe["shrout"] = pd.to_numeric(crsp_universe["shrout"], errors="coerce")
crsp_universe["me"] = crsp_universe["prc"].abs() * crsp_universe["shrout"] / 1000
crsp_universe["ym"] = crsp_universe["date"].dt.to_period("M")
crsp_universe["siccd"] = pd.to_numeric(crsp_universe["siccd"], errors="coerce")

print(f"✅ CRSP universe: {len(crsp_universe):,} firm-months, "
      f"{crsp_universe['permno'].nunique():,} firms")

# ── Assign FF48 industry using Ken French's SIC ranges ───────────────────────
# Simplified but accurate mapping using 2-digit SIC + key sub-ranges
def sic_to_ff48(sic):
    """Map 4-digit SIC to Fama-French 48 industry number."""
    if pd.isna(sic):
        return 0
    s = int(sic)
    # Food Products
    if (100 <= s <= 299) or (700 <= s <= 799) or (2000 <= s <= 2046) or \
       (2048 <= s <= 2099) or (910 <= s <= 919):
        return 1
    # Candy & Soda
    if (2064 <= s <= 2068) or (2086 <= s <= 2087) or (2096 <= s <= 2097):
        return 2
    # Beer
    if 2080 <= s <= 2085:
        return 3
    # Smoke
    if 2100 <= s <= 2199:
        return 4
    # Toys
    if s in [3930,3931] or (3940 <= s <= 3949) or (920 <= s <= 999):
        return 5
    # Fun (Entertainment)
    if (7800 <= s <= 7833) or (7840 <= s <= 7841) or (7900 <= s <= 7999):
        return 6
    # Books
    if (2700 <= s <= 2749) or (2770 <= s <= 2771) or (2780 <= s <= 2799):
        return 7
    # Household
    if s == 2047 or (2510 <= s <= 2519) or (2590 <= s <= 2599) or \
       (2840 <= s <= 2844) or (3160 <= s <= 3199) or (3630 <= s <= 3639) or \
       (3750 <= s <= 3751) or (3860 <= s <= 3879) or (3910 <= s <= 3919) or \
       (3960 <= s <= 3962) or (3991 <= s <= 3991) or (3995 <= s <= 3995):
        return 8
    # Clothes (Apparel)
    if (2300 <= s <= 2390) or (3020 <= s <= 3021) or (3100 <= s <= 3151) or \
       (3963 <= s <= 3965):
        return 9
    # Health (MedEq)
    if s == 3693 or (3840 <= s <= 3851):
        return 10
    # Drugs (Pharma)
    if 2830 <= s <= 2836:
        return 11
    # Chemicals
    if (2800 <= s <= 2829) or (2850 <= s <= 2899):
        return 12
    # Textiles
    if 2200 <= s <= 2299:
        return 13
    # Construction Materials
    if (800 <= s <= 899) or (2400 <= s <= 2499) or (2660 <= s <= 2661) or \
       (2950 <= s <= 2952) or (3200 <= s <= 3299) or (3420 <= s <= 3442) or \
       (3446 <= s <= 3452) or (3490 <= s <= 3499) or (3996 <= s <= 3996):
        return 14
    # Steel
    if 3300 <= s <= 3399:
        return 15
    # Fabricated Products / Machinery
    if (3400 <= s <= 3400) or (3443 <= s <= 3444) or (3460 <= s <= 3479) or \
       (3510 <= s <= 3536) or (3538 <= s <= 3599):
        return 16
    # Electrical Equipment
    if (3600 <= s <= 3621) or (3623 <= s <= 3629) or (3640 <= s <= 3646) or \
       (3648 <= s <= 3649) or (3660 <= s <= 3660) or (3690 <= s <= 3692) or \
       (3699 <= s <= 3699):
        return 17
    # Autos
    if (2296 <= s <= 2296) or (2396 <= s <= 2396) or (3010 <= s <= 3011) or \
       (3537 <= s <= 3537) or (3647 <= s <= 3647) or (3694 <= s <= 3694) or \
       (3700 <= s <= 3716) or (3790 <= s <= 3792) or (3799 <= s <= 3799):
        return 18
    # Aero (Aircraft)
    if (3720 <= s <= 3729) or (3760 <= s <= 3769) or (3795 <= s <= 3795):
        return 19
    # Ships
    if (3730 <= s <= 3731) or (3740 <= s <= 3743):
        return 20
    # Guns (Defense)
    if 3480 <= s <= 3489:
        return 21
    # Gold
    if 1040 <= s <= 1049:
        return 22
    # Mines
    if (1000 <= s <= 1039) or (1050 <= s <= 1099) or (1100 <= s <= 1119) or \
       (1400 <= s <= 1499):
        return 23
    # Coal
    if 1200 <= s <= 1299:
        return 24
    # Oil (Petroleum)
    if (1300 <= s <= 1389) or (2900 <= s <= 2912) or (2990 <= s <= 2999):
        return 25
    # Utilities
    if (4900 <= s <= 4942) or (4950 <= s <= 4991):
        return 26
    # Telecom
    if 4800 <= s <= 4899:
        return 27
    # Personal Services
    if (7020 <= s <= 7021) or (7030 <= s <= 7033) or (7200 <= s <= 7299) or \
       (7600 <= s <= 7699):
        return 28
    # Business Services
    if (2750 <= s <= 2759) or (7300 <= s <= 7399) or (7380 <= s <= 7399) or \
       (8700 <= s <= 8748) or (8900 <= s <= 8999):
        return 29
    # Computers (Hardware)
    if (3570 <= s <= 3579) or (3680 <= s <= 3689) or (3695 <= s <= 3695):
        return 30
    # Computer Software
    if 7370 <= s <= 7379:
        return 31
    # Electronic Equipment (Chips)
    if (3622 <= s <= 3622) or (3661 <= s <= 3669) or (3670 <= s <= 3679) or \
       (3810 <= s <= 3812):
        return 32
    # Measuring & Control Equipment (LabEq)
    if 3811 <= s <= 3839:
        return 33
    # Paper Products
    if (2520 <= s <= 2549) or (2600 <= s <= 2659) or (2670 <= s <= 2699):
        return 34
    # Shipping Containers (Boxes)
    if (2440 <= s <= 2449) or (2640 <= s <= 2659) or (3085 <= s <= 3089) or \
       (3410 <= s <= 3412):
        return 35
    # Transportation
    if (4000 <= s <= 4099) or (4100 <= s <= 4199) or (4200 <= s <= 4299) or \
       (4400 <= s <= 4499) or (4500 <= s <= 4599) or (4600 <= s <= 4699) or \
       (4700 <= s <= 4799):
        return 36
    # Wholesale
    if 5000 <= s <= 5199:
        return 37
    # Retail
    if 5200 <= s <= 5799 or (5900 <= s <= 5999):
        return 38
    # Restaurants & Hotels (Meals)
    if (5800 <= s <= 5813) or (s == 5890) or (7000 <= s <= 7019) or \
       (7040 <= s <= 7049) or (s == 7213):
        return 39
    # Banking
    if 6000 <= s <= 6199:
        return 40
    # Insurance
    if 6200 <= s <= 6411:
        return 41
    # Real Estate
    if 6500 <= s <= 6553:
        return 42
    # Trading (Other Finance)
    if (6700 <= s <= 6799) or (6200 <= s <= 6299):
        return 43
    # Construction
    if 1500 <= s <= 1799:
        return 44
    # Healthcare
    if (8000 <= s <= 8099):
        return 45
    # Other
    return 0

crsp_universe["ff48"] = crsp_universe["siccd"].apply(sic_to_ff48)

# Lagged ME for VW
crsp_universe = crsp_universe.sort_values(["permno", "date"])
crsp_universe["me_lag"] = crsp_universe.groupby("permno")["me"].shift(1)

# VW industry returns by month × FF48
def vw_ind_ret(grp):
    sub = grp.dropna(subset=["ret", "me_lag"])
    if sub.empty or sub["me_lag"].sum() == 0:
        return np.nan
    w = sub["me_lag"] / sub["me_lag"].sum()
    return (w * sub["ret"]).sum()

print("⏳ Computing VW industry returns (this may take a minute)…")
ind_ret = crsp_universe.groupby(["ym", "ff48"]).apply(vw_ind_ret).rename("ind_ret")
ind_ret = ind_ret.reset_index()
print(f"✅ Constructed VW returns for {ind_ret['ff48'].nunique()} FF48 industries")

# ── Assign FF48 codes to portfolio stocks ────────────────────────────────────
port_ind = crsp_port.dropna(subset=["ret_adj"]).copy()
port_ind["siccd"] = pd.to_numeric(port_ind["siccd"], errors="coerce")
port_ind["ff48"] = port_ind["siccd"].apply(sic_to_ff48)

# Merge industry returns
port_ind = port_ind.merge(ind_ret, on=["ym", "ff48"], how="left")
port_ind["ret_ind_adj"] = port_ind["ret_adj"] - port_ind["ind_ret"]

# Show coverage
n_matched = port_ind["ind_ret"].notna().sum()
n_total = len(port_ind)
print(f"\nIndustry return matched: {n_matched}/{n_total} ({n_matched/n_total*100:.1f}%)")

# ── Industry-adjusted portfolio returns (EW) ─────────────────────────────────
ind_ew = port_ind.groupby("ym")["ret_ind_adj"].mean().rename("ret_ind_ew")

# ── Industry-adjusted portfolio returns (VW) ─────────────────────────────────
port_ind = port_ind.merge(
    crsp_sorted[["permno", "date", "me_lag"]],
    on=["permno", "date"],
    how="left",
    suffixes=("", "_v2")
)

def vw_ind_adj_ret(grp):
    sub = grp.dropna(subset=["ret_ind_adj", "me_lag"])
    if sub.empty or sub["me_lag"].sum() == 0:
        return np.nan
    w = sub["me_lag"] / sub["me_lag"].sum()
    return (w * sub["ret_ind_adj"]).sum()

ind_vw = port_ind.groupby("ym").apply(vw_ind_adj_ret).rename("ret_ind_vw")

# Merge into reg_df
reg_df = reg_df.join(ind_ew, how="left")
reg_df = reg_df.join(ind_vw, how="left")

print("\n── Industry-Adjusted Returns ──")
print(f"  EW mean: {reg_df['ret_ind_ew'].mean()*100:.3f}% (Paper: 0.40%)")
print(f"  VW mean: {reg_df['ret_ind_vw'].mean()*100:.3f}% (Paper: 0.37%)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8b · Industry-adjusted alpha (Newey–West t-stat on the mean)
# ═══════════════════════════════════════════════════════════════════════════════

print("="*70)
print("  BENCHMARK: Industry-Adjusted Returns")
print("="*70)

# EW industry-adjusted
ew_ind = run_regression(
    y=reg_df["ret_ind_ew"],
    X=pd.DataFrame(index=reg_df.index),
    model_name="EW Portfolio — Industry-Adjusted (Paper: α=0.40%, t=3.36)"
)

# VW industry-adjusted
vw_ind = run_regression(
    y=reg_df["ret_ind_vw"],
    X=pd.DataFrame(index=reg_df.index),
    model_name="VW Portfolio — Industry-Adjusted (Paper: α=0.37%, t=2.46)"
)

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY: All benchmarks
# ═══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "═"*90)
print("  COMPLETE TABLE 4 REPLICATION: Portfolio II (Feb 1998 – Dec 2009)")
print("═"*90)

final_comparison = []
for label, ew_model, vw_model, paper_ew_a, paper_ew_t, paper_vw_a, paper_vw_t in [
    ("Risk-free",  ew_rf,   vw_rf,   0.44, 2.89, 0.32, 1.65),
    ("Industry",   ew_ind,  vw_ind,  0.40, 3.36, 0.37, 2.46),
    ("Mkt-RF",     ew_capm, vw_capm, 0.49, 3.33, 0.26, 1.35),
    ("4-Factor",   ew_4f,   vw_4f,   0.34, 2.44, 0.18, 0.95),
]:
    ew_a = ew_model.params["const"] * 100
    ew_t = ew_model.tvalues["const"]
    vw_a = vw_model.params["const"] * 100
    vw_t = vw_model.tvalues["const"]
    final_comparison.append({
        "Benchmark":     label,
        "EW α (%)":      f"{ew_a:6.2f}",
        "EW t":          f"{ew_t:5.2f}",
        "Paper EW α":    f"{paper_ew_a:5.2f}",
        "Paper EW t":    f"{paper_ew_t:5.2f}",
        "  |  VW α (%)": f"{vw_a:6.2f}",
        "VW t":          f"{vw_t:5.2f}",
        "Paper VW α":    f"{paper_vw_a:5.2f}",
        "Paper VW t":    f"{paper_vw_t:5.2f}",
    })

final_df = pd.DataFrame(final_comparison)
print()
print(final_df.to_string(index=False))
print()
print("═"*90)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9 · Export results & disconnect
# ═══════════════════════════════════════════════════════════════════════════════

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# ── Save portfolio returns ───────────────────────────────────────────────────
port_out = portfolios.copy()
port_out.index = port_out.index.to_timestamp()
port_out.to_csv(f"{output_dir}/edmans_portfolio_II_returns.csv")

# ── Save company mapping ────────────────────────────────────────────────────
df_companies.to_csv(f"{output_dir}/edmans_1998_company_mapping.csv", index=False)

# ── Save regression comparison ──────────────────────────────────────────────
final_df.to_csv(f"{output_dir}/edmans_table4_replication.csv", index=False)

print("✅ Results saved to ./output/")
print("   • edmans_portfolio_II_returns.csv")
print("   • edmans_1998_company_mapping.csv")
print("   • edmans_table4_replication.csv")

# ── Close WRDS connection ────────────────────────────────────────────────────
db.close()
print("\n🔒 WRDS connection closed.")